In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Check GPU
!nvidia-smi

# Clone CIRI-FS
%cd /content
!rm -rf CIRI-FS

!git clone --branch asal/CiriEXT4 https://github.com/isusbu/CIRI-FS.git

%cd /content/CIRI-FS

# Verify the correct branch
!git branch --show-current

Tue Aug  4 20:10:22 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [8]:
!find /content/drive -type d | grep "shortcut-targets"

/content/drive/.shortcut-targets-by-id


In [11]:
import os

DRIVE_DATA = "/content/drive/.shortcut-targets-by-id/1xrerJFYBKPY28WmDN6o8zkkE73mSAHn0/Dataset/EXT4_XML_Dataset"

In [12]:
!cp -r "$DRIVE_DATA/ext4_SD" icse25_data/datasets/synthesize_config/
!cp -r "$DRIVE_DATA/ext4_CPD" icse25_data/datasets/synthesize_config/
!cp -r "$DRIVE_DATA/ext4_CCD" icse25_data/datasets/synthesize_config/

In [13]:
!mkdir -p icse25_data/datasets/synthesize_config/FewShots

In [14]:
!cp -r "$DRIVE_DATA/FewShots/ext4_SD" \
icse25_data/datasets/synthesize_config/FewShots/

!cp -r "$DRIVE_DATA/FewShots/ext4_CPD" \
icse25_data/datasets/synthesize_config/FewShots/

!cp -r "$DRIVE_DATA/FewShots/ext4_CCD" \
icse25_data/datasets/synthesize_config/FewShots/

In [15]:
!find icse25_data/datasets/synthesize_config/FewShots -maxdepth 2

icse25_data/datasets/synthesize_config/FewShots
icse25_data/datasets/synthesize_config/FewShots/ext4_CPD
icse25_data/datasets/synthesize_config/FewShots/ext4_CPD/Misconfig
icse25_data/datasets/synthesize_config/FewShots/ext4_CPD/ValidConfig
icse25_data/datasets/synthesize_config/FewShots/ext4_SD
icse25_data/datasets/synthesize_config/FewShots/ext4_SD/Misconfig
icse25_data/datasets/synthesize_config/FewShots/ext4_SD/ValidConfig
icse25_data/datasets/synthesize_config/FewShots/ext4_CCD
icse25_data/datasets/synthesize_config/FewShots/ext4_CCD/Misconfig
icse25_data/datasets/synthesize_config/FewShots/ext4_CCD/ValidConfig


In [17]:
!pip install -q --upgrade \
    "bitsandbytes>=0.46.1" \
    accelerate \
    transformers \
    sentencepiece \
    anthropic \
    "protobuf>=5.29.1,<6"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 118.7 MB/s eta 0:00:00


In [18]:
import bitsandbytes as bnb
import transformers
import accelerate
import torch

print("bitsandbytes:", bnb.__version__)
print("transformers:", transformers.__version__)
print("accelerate:", accelerate.__version__)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

bitsandbytes: 0.50.0
transformers: 5.14.1
accelerate: 1.14.0
PyTorch: 2.11.0+cu128
CUDA available: True


In [19]:
!python -m ciri.ciri_eng \
    --input_path \
        icse25_data/datasets/synthesize_config/ext4_SD/erroneous \
    --output_path \
        icse25_data/results/synthesize_config/ext4_SD/deepseek-coder-6.7b-instruct/few_shot/erroneous \
    --model deepseek-coder-6.7b-instruct \
    --system ext4 \
    --version 1.47.0 \
    --validconfig_shot_num 1 \
    --misconfig_shot_num 3 \
    --shot_selection random \
    --file_format xml \
    --verbose

2026-08-04 20:55:56 - Ciri - INFO - Using device: CUDA
2026-08-04 20:55:56 - Ciri - INFO - Using dtype: torch.bfloat16
2026-08-04 20:55:56 - Ciri - INFO - Loading DeepSeek model: deepseek-ai/deepseek-coder-6.7b-instruct
model.safetensors.index.json: 100% 25.1k/25.1k [00:00<00:00, 53.9MB/s]
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0% 0/2 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0% 0.00/9.98G [00:00<?, ?B/s]         
Reconstructing (incomplete total...):  15% 2.03G/13.5G [00:29<01:08, 168MB/s, 36.9MB/s  ]
Reconstructing (incomplete total...):  25% 3.38G/13.5G [01:09<03:23, 49.6MB/s, 42.6MB/s  ]
Reconstructing (incomplete total...):  32% 4.28G/13.5G [01:39<04:00, 38.2MB/s, 33.2MB/s  ]
Reconstructing (incomplete total...):  35% 4.72G/13.5G [01:59<04:46, 30.6MB/s, 35.8MB/s  ]
Reconstructing (incomplete total...):  39% 5.28G/13.5G [02:19<04:40, 29.2MB/s, 40.7MB/s  ]
Reconstructing (incomplete total...):  47% 6.38G/13.

In [20]:
!python -m ciri.ciri_eng \
    --input_path \
        icse25_data/datasets/synthesize_config/ext4_SD/correct \
    --output_path \
        icse25_data/results/synthesize_config/ext4_SD/deepseek-coder-6.7b-instruct/few_shot/correct \
    --model deepseek-coder-6.7b-instruct \
    --system ext4 \
    --version 1.47.0 \
    --validconfig_shot_num 1 \
    --misconfig_shot_num 3 \
    --shot_selection random \
    --file_format xml \
    --verbose

2026-08-04 21:11:09 - Ciri - INFO - Using device: CUDA
2026-08-04 21:11:09 - Ciri - INFO - Using dtype: torch.bfloat16
2026-08-04 21:11:09 - Ciri - INFO - Loading DeepSeek model: deepseek-ai/deepseek-coder-6.7b-instruct
Loading weights: 100% 291/291 [00:55<00:00,  5.26it/s]
2026-08-04 21:12:10 - Ciri - INFO - Model loaded successfully on CUDA!
2026-08-04 21:12:10 - Ciri - INFO - Selected misconfiguration shots: [15, 6, 16]
2026-08-04 21:12:10 - Ciri - INFO - Selected valid configuration shots: [1]
2026-08-04 21:12:10 - Ciri - INFO - [llm_gen] Using device: CUDA
[Ciri] Start
[Ciri] Running for file icse25_data/results/synthesize_config/ext4_SD/deepseek-coder-6.7b-instruct/few_shot/correct/4
[Ciri] Result: The CONFIGURATION FILE IS CORRECT
[Ciri] Writing log file to icse25_data/results/synthesize_config/ext4_SD/deepseek-coder-6.7b-instruct/few_shot/correct/4
[Ciri] End
2026-08-04 21:13:03 - Ciri - INFO - Selected misconfiguration shots: [1, 5, 14]
2026-08-04 21:13:03 - Ciri - INFO - Sele

In [21]:
!python icse25_data/script/result_parser.py \
    --project ext4_SD \
    --model deepseek-coder-6.7b-instruct \
    --mode few_shot

[Ciri Result] on ext4_SD with deepseek-coder-6.7b-instruct and few_shot mode
File-Level: Precision: 1.00, Recall: 0.20, Accuracy: 0.60, F1: 0.33
Param-Level: Precision: 0.33, Recall: 0.20, Accuracy: 0.93, F1: 0.25


CPD

In [22]:
!cp "$DRIVE_DATA/Groundtruth/ext4_CPD.tsv" \
    icse25_data/datasets/synthesize_config/ground_truth/

In [23]:
!python -m ciri.ciri_eng \
    --input_path \
        icse25_data/datasets/synthesize_config/ext4_CPD/erroneous \
    --output_path \
        icse25_data/results/synthesize_config/ext4_CPD/deepseek-coder-6.7b-instruct/few_shot/erroneous \
    --model deepseek-coder-6.7b-instruct \
    --system ext4 \
    --version 1.47.0 \
    --validconfig_shot_num 1 \
    --misconfig_shot_num 3 \
    --shot_selection random \
    --file_format xml \
    --verbose

2026-08-04 21:26:11 - Ciri - INFO - Using device: CUDA
2026-08-04 21:26:11 - Ciri - INFO - Using dtype: torch.bfloat16
2026-08-04 21:26:11 - Ciri - INFO - Loading DeepSeek model: deepseek-ai/deepseek-coder-6.7b-instruct
Loading weights: 100% 291/291 [00:55<00:00,  5.25it/s]
2026-08-04 21:27:11 - Ciri - INFO - Model loaded successfully on CUDA!
2026-08-04 21:27:11 - Ciri - INFO - Selected misconfiguration shots: [11, 3, 6]
2026-08-04 21:27:11 - Ciri - INFO - Selected valid configuration shots: [3]
2026-08-04 21:27:11 - Ciri - INFO - [llm_gen] Using device: CUDA
[Ciri] Start
[Ciri] Running for file icse25_data/results/synthesize_config/ext4_CPD/deepseek-coder-6.7b-instruct/few_shot/erroneous/4
[Ciri] Result: The CONFIGURATION FILE IS CORRECT
[Ciri] Writing log file to icse25_data/results/synthesize_config/ext4_CPD/deepseek-coder-6.7b-instruct/few_shot/erroneous/4
[Ciri] End
2026-08-04 21:29:10 - Ciri - INFO - Selected misconfiguration shots: [7, 4, 2]
2026-08-04 21:29:10 - Ciri - INFO - 

In [24]:
!python -m ciri.ciri_eng \
    --input_path \
        icse25_data/datasets/synthesize_config/ext4_CPD/correct \
    --output_path \
        icse25_data/results/synthesize_config/ext4_CPD/deepseek-coder-6.7b-instruct/few_shot/correct \
    --model deepseek-coder-6.7b-instruct \
    --system ext4 \
    --version 1.47.0 \
    --validconfig_shot_num 1 \
    --misconfig_shot_num 3 \
    --shot_selection random \
    --file_format xml \
    --verbose

2026-08-04 21:34:30 - Ciri - INFO - Using device: CUDA
2026-08-04 21:34:30 - Ciri - INFO - Using dtype: torch.bfloat16
2026-08-04 21:34:30 - Ciri - INFO - Loading DeepSeek model: deepseek-ai/deepseek-coder-6.7b-instruct
Loading weights: 100% 291/291 [00:55<00:00,  5.26it/s]
2026-08-04 21:35:30 - Ciri - INFO - Model loaded successfully on CUDA!
2026-08-04 21:35:30 - Ciri - INFO - Selected misconfiguration shots: [5, 14, 7]
2026-08-04 21:35:30 - Ciri - INFO - Selected valid configuration shots: [4]
2026-08-04 21:35:30 - Ciri - INFO - [llm_gen] Using device: CUDA
[Ciri] Start
[Ciri] Running for file icse25_data/results/synthesize_config/ext4_CPD/deepseek-coder-6.7b-instruct/few_shot/correct/4
[Ciri] Result: The CONFIGURATION FILE IS CORRECT
[Ciri] Writing log file to icse25_data/results/synthesize_config/ext4_CPD/deepseek-coder-6.7b-instruct/few_shot/correct/4
[Ciri] End
2026-08-04 21:36:35 - Ciri - INFO - Selected misconfiguration shots: [2, 12, 1]
2026-08-04 21:36:35 - Ciri - INFO - Sel

In [25]:
!python icse25_data/script/result_parser.py \
    --project ext4_CPD \
    --model deepseek-coder-6.7b-instruct \
    --mode few_shot

[Ciri Result] on ext4_CPD with deepseek-coder-6.7b-instruct and few_shot mode
File-Level: Precision: N.A., Recall: 0.00, Accuracy: 0.50, F1: N.A.
Param-Level: Precision: N.A., Recall: 0.00, Accuracy: 0.93, F1: N.A.


CCD

In [26]:
!cp "$DRIVE_DATA/Groundtruth/ext4_CCD.tsv" \
    icse25_data/datasets/synthesize_config/ground_truth/

In [27]:
!python -m ciri.ciri_eng \
    --input_path \
        icse25_data/datasets/synthesize_config/ext4_CCD/erroneous \
    --output_path \
        icse25_data/results/synthesize_config/ext4_CCD/deepseek-coder-6.7b-instruct/few_shot/erroneous \
    --model deepseek-coder-6.7b-instruct \
    --system ext4 \
    --version 1.47.0 \
    --validconfig_shot_num 1 \
    --misconfig_shot_num 3 \
    --shot_selection random \
    --file_format xml \
    --verbose

2026-08-04 21:44:09 - Ciri - INFO - Using device: CUDA
2026-08-04 21:44:09 - Ciri - INFO - Using dtype: torch.bfloat16
2026-08-04 21:44:09 - Ciri - INFO - Loading DeepSeek model: deepseek-ai/deepseek-coder-6.7b-instruct
Loading weights: 100% 291/291 [00:55<00:00,  5.25it/s]
2026-08-04 21:45:10 - Ciri - INFO - Model loaded successfully on CUDA!
2026-08-04 21:45:10 - Ciri - INFO - Selected misconfiguration shots: [6, 1, 3]
2026-08-04 21:45:10 - Ciri - INFO - Selected valid configuration shots: [1]
2026-08-04 21:45:10 - Ciri - INFO - [llm_gen] Using device: CUDA
[Ciri] Start
[Ciri] Running for file icse25_data/results/synthesize_config/ext4_CCD/deepseek-coder-6.7b-instruct/few_shot/erroneous/12
[Ciri] Result: The CONFIGURATION FILE IS CORRECT
[Ciri] Writing log file to icse25_data/results/synthesize_config/ext4_CCD/deepseek-coder-6.7b-instruct/few_shot/erroneous/12
[Ciri] End
2026-08-04 21:45:56 - Ciri - INFO - Selected misconfiguration shots: [8, 10, 14]
2026-08-04 21:45:56 - Ciri - INFO

In [30]:
!python -m ciri.ciri_eng \
    --input_path \
        icse25_data/datasets/synthesize_config/ext4_CCD/correct \
    --output_path \
        icse25_data/results/synthesize_config/ext4_CCD/deepseek-coder-6.7b-instruct/few_shot/correct \
    --model deepseek-coder-6.7b-instruct \
    --system ext4 \
    --version 1.47.0 \
    --validconfig_shot_num 1 \
    --misconfig_shot_num 3 \
    --shot_selection random \
    --file_format xml \
    --verbose

2026-08-04 23:27:34 - Ciri - INFO - Using device: CUDA
2026-08-04 23:27:34 - Ciri - INFO - Using dtype: torch.bfloat16
2026-08-04 23:27:34 - Ciri - INFO - Loading DeepSeek model: deepseek-ai/deepseek-coder-6.7b-instruct
Loading weights: 100% 291/291 [00:56<00:00,  5.17it/s]
2026-08-04 23:28:37 - Ciri - INFO - Model loaded successfully on CUDA!
2026-08-04 23:28:37 - Ciri - INFO - Selected misconfiguration shots: [7, 11, 10]
2026-08-04 23:28:37 - Ciri - INFO - Selected valid configuration shots: [2]
2026-08-04 23:28:37 - Ciri - INFO - [llm_gen] Using device: CUDA
[Ciri] Start
[Ciri] Running for file icse25_data/results/synthesize_config/ext4_CCD/deepseek-coder-6.7b-instruct/few_shot/correct/12
[Ciri] Result: The CONFIGURATION FILE IS CORRECT
[Ciri] Writing log file to icse25_data/results/synthesize_config/ext4_CCD/deepseek-coder-6.7b-instruct/few_shot/correct/12
[Ciri] End
2026-08-04 23:29:37 - Ciri - INFO - Selected misconfiguration shots: [1, 10, 5]
2026-08-04 23:29:37 - Ciri - INFO - 

In [31]:
!python icse25_data/script/result_parser.py \
    --project ext4_CCD \
    --model deepseek-coder-6.7b-instruct \
    --mode few_shot

[Ciri Result] on ext4_CCD with deepseek-coder-6.7b-instruct and few_shot mode
File-Level: Precision: N.A., Recall: 0.00, Accuracy: 0.50, F1: N.A.
Param-Level: Precision: N.A., Recall: 0.00, Accuracy: 0.94, F1: N.A.


In [33]:
!mkdir -p "/content/drive/MyDrive/CIRI_EXT4/DeepSeek_results"

!cp -r icse25_data/results/synthesize_config/ext4_SD \
      "/content/drive/MyDrive/CIRI_EXT4/DeepSeek_results/"

!cp -r icse25_data/results/synthesize_config/ext4_CPD \
      "/content/drive/MyDrive/CIRI_EXT4/DeepSeek_results/"

!cp -r icse25_data/results/synthesize_config/ext4_CCD \
      "/content/drive/MyDrive/CIRI_EXT4/DeepSeek_results/"